In [20]:
import pandas as pd
import numpy as np 
import sklearn
import lightgbm
import statsmodels
import matplotlib


train_df = pd.read_json('../data/train.json')
train_df.head()

# Снимаем ограничения на максимальное количество выводимых колонок и строк
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# Округлим вывод чисел в таблице до 3 знаков после запятой для читаемости
pd.set_option('display.float_format', '{:.3f}'.format)


In [21]:
train_df.shape

(49352, 15)

In [22]:
print(train_df.corr(numeric_only=True))

            bathrooms  bedrooms  latitude  listing_id  longitude  price
bathrooms       1.000     0.533    -0.010       0.001      0.010  0.070
bedrooms        0.533     1.000    -0.005       0.012      0.007  0.052
latitude       -0.010    -0.005     1.000       0.002     -0.967 -0.001
listing_id      0.001     0.012     0.002       1.000     -0.001  0.008
longitude       0.010     0.007    -0.967      -0.001      1.000 -0.000
price           0.070     0.052    -0.001       0.008     -0.000  1.000


In [23]:
train_df.info()

<class 'pandas.DataFrame'>
Index: 49352 entries, 4 to 124009
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   bathrooms        49352 non-null  float64
 1   bedrooms         49352 non-null  int64  
 2   building_id      49352 non-null  str    
 3   created          49352 non-null  str    
 4   description      49352 non-null  str    
 5   display_address  49352 non-null  str    
 6   features         49352 non-null  object 
 7   latitude         49352 non-null  float64
 8   listing_id       49352 non-null  int64  
 9   longitude        49352 non-null  float64
 10  manager_id       49352 non-null  str    
 11  photos           49352 non-null  object 
 12  price            49352 non-null  int64  
 13  street_address   49352 non-null  str    
 14  interest_level   49352 non-null  str    
dtypes: float64(3), int64(3), object(2), str(7)
memory usage: 6.0+ MB


In [24]:
train_df['created']

4         2016-06-16 05:55:27
6         2016-06-01 05:44:33
9         2016-06-14 15:19:59
10        2016-06-24 07:54:24
15        2016-06-28 03:50:23
16        2016-06-28 05:59:06
18        2016-06-08 06:21:36
19        2016-06-05 05:28:22
23        2016-06-09 04:42:03
32        2016-06-28 03:26:18
33        2016-06-04 02:21:27
36        2016-06-01 02:51:22
38        2016-06-25 05:28:30
39        2016-06-11 03:46:53
42        2016-06-11 04:24:39
43        2016-06-20 10:14:52
44        2016-06-11 03:59:20
46        2016-06-16 07:43:48
49        2016-06-16 03:23:57
61        2016-06-25 01:35:36
66        2016-06-24 02:16:54
67        2016-06-10 02:44:56
69        2016-06-14 02:28:02
74        2016-06-02 02:37:02
78        2016-06-09 05:38:52
80        2016-06-20 19:09:31
82        2016-06-05 11:31:08
83        2016-06-29 04:52:59
84        2016-06-29 02:40:31
85        2016-06-01 03:38:31
87        2016-06-25 06:08:16
88        2016-06-16 03:11:45
89        2016-06-08 06:18:46
92        

# Разделяем created на месяц, день, день недели и час

In [25]:
train_df['created'] = pd.to_datetime(train_df['created'])

train_df['month'] = train_df['created'].dt.month
train_df['day'] = train_df['created'].dt.day
train_df['day_of_week'] = train_df['created'].dt.dayofweek
train_df['hour'] = train_df['created'].dt.hour

print(train_df[['created','month','day','day_of_week','hour']].head())




               created  month  day  day_of_week  hour
4  2016-06-16 05:55:27      6   16            3     5
6  2016-06-01 05:44:33      6    1            2     5
9  2016-06-14 15:19:59      6   14            1    15
10 2016-06-24 07:54:24      6   24            4     7
15 2016-06-28 03:50:23      6   28            1     3


In [26]:
print(train_df['interest_level'].value_counts())

interest_level
low       34284
medium    11229
high       3839
Name: count, dtype: int64


# Переводим в проценты 

In [27]:
print((train_df['interest_level'].value_counts(normalize=True) * 100).round(1))

interest_level
low      69.500
medium   22.800
high      7.800
Name: proportion, dtype: float64


# Заменяем слова на цифры с map  
### компьютер не воспринимает слова, соответсвенно для дальнейших операций с interest_level нужно перевести его в цифры

In [28]:
mapping = {'low': 0, 'medium': 1, 'high': 2}

train_df['interest_level_num'] = train_df['interest_level'].map(mapping)

print(train_df[['interest_level', 'interest_level_num']].head())


   interest_level  interest_level_num
4          medium                   1
6             low                   0
9          medium                   1
10         medium                   1
15            low                   0


In [29]:
train_df.info()

<class 'pandas.DataFrame'>
Index: 49352 entries, 4 to 124009
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   bathrooms           49352 non-null  float64       
 1   bedrooms            49352 non-null  int64         
 2   building_id         49352 non-null  str           
 3   created             49352 non-null  datetime64[us]
 4   description         49352 non-null  str           
 5   display_address     49352 non-null  str           
 6   features            49352 non-null  object        
 7   latitude            49352 non-null  float64       
 8   listing_id          49352 non-null  int64         
 9   longitude           49352 non-null  float64       
 10  manager_id          49352 non-null  str           
 11  photos              49352 non-null  object        
 12  price               49352 non-null  int64         
 13  street_address      49352 non-null  str           
 14  inter

In [30]:
df = train_df[['bathrooms', 'bedrooms', 'interest_level_num', 'price']]

df.head()

,bathrooms,bedrooms,interest_level_num,price
4,1.000,1,1,2400
6,1.000,2,0,3800
9,1.000,2,1,3495
10,1.500,3,1,3000
15,1.000,0,0,2795


In [31]:
print(train_df['price'].describe())

count     49352.000
mean       3830.174
std       22066.866
min          43.000
25%        2500.000
50%        3150.000
75%        4100.000
max     4490000.000
Name: price, dtype: float64


# Добавляем признаки

In [33]:
# Создаем квадрат для количества ванных комнат
train_df['bathrooms_squared'] = train_df['bathrooms'] ** 2

# 2. Создаем квадрат для количества спален
train_df['bedrooms_squared'] = train_df['bedrooms'] ** 2

# 3. Создаём квадрат для цены
train_df['price_square'] = train_df['price'] ** 2

# 4. Создаём квадрат для объединёных ванных с спальнями. 
train_df['total_rooms'] = train_df['bathrooms'] + train_df['bedrooms']

# 5. Добавляем соотношение ванных к спальням
# Добавляем 0.1, чтобы случайно не разделилось на ноль, если спален 0
train_df['bath_to_bed_ratio'] = train_df['bathrooms'] / (train_df['bedrooms'] + 0.1)

# 6. Длина описания объявления
train_df['desc_length'] = train_df['description'].str.len().fillna(0)


# Проверяем, что новые столбцы успешно добавились
display(train_df[['bathrooms', 'bathrooms_squared', 'bedrooms', 'bedrooms_squared', 'total_rooms', 'price_square', 'bath_to_bed_ratio', 'desc_length']].head())


,bathrooms,bathrooms_squared,bedrooms,bedrooms_squared,total_rooms,price_square,bath_to_bed_ratio,desc_length
4,1.000,1.000,1,1,2.000,5760000,0.909,553
6,1.000,1.000,2,4,3.000,14440000,0.476,827
9,1.000,1.000,2,4,3.000,12215025,0.476,799
10,1.500,2.250,3,9,4.500,9000000,0.484,588
15,1.000,1.000,0,0,1.000,7812025,10.000,344
